# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:38<00:00, 12.74s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Certified Refurb HP EliteBook 830 G7 10th-Gen. i7 13.3" Touch Laptop for $321 + free shipping\nDetails: It\'s the best price we could find by $69. It includes a 2-year Allstate warranty. Buy Now at eBay\nFeatures: \nURL: https://www.dealnews.com/products/HP/HP-Elite-Book-830-G7-10-th-Gen-i7-13-3-Touch-Laptop/497337.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Gamegaga BSP-D11 Telescopic Mobile Gaming Controller for $10 + free shipping w/ first order
Details: That's a savings of $2, though most other places charge closer to $30 or more for this mobile controller. Of note, this purchase will also garner $2.36 in import & payment processing fees.New Alibaba customers get free shipping; otherwise, it starts at around $12. Buy Now at Ali

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Oukitel P2001 Plus is a high-capacity portable power station built around a LiFePO4 battery and integrated BMS for long cycle life and safety. It delivers a continuous 2,400W AC output with 4,800W peak surge capability, supports up to 1,800W AC input charging and 500W max solar input, and includes multiple output ports for powering appliances, tools, and electronics during outages or off-grid use.', price=569.0, url='https://www.dealnews.com/Oukitel-P2001-Plus-Portable-Power-Station-2048-Wh-2400-W-for-569-free-shipping/21803780.html?iref=rss-c142'), Deal(product_description='Eco-Worthy’s 48V 100Ah server-rack lithium battery comes as a six-pack delivering about 30.72 kWh of usable energy in a standard 3U-compatible full-metal enclosure. Each battery pack includes a 100A battery management system, wireless connectivity and smart monitoring features for system diagnostics and safety, and multi-layer protection intended for off-grid, backup, 

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Oukitel P2001 Plus is a high-capacity portable power station built around a LiFePO4 battery and integrated BMS for long cycle life and safety. It delivers a continuous 2,400W AC output with 4,800W peak surge capability, supports up to 1,800W AC input charging and 500W max solar input, and includes multiple output ports for powering appliances, tools, and electronics during outages or off-grid use.
569.0
https://www.dealnews.com/Oukitel-P2001-Plus-Portable-Power-Station-2048-Wh-2400-W-for-569-free-shipping/21803780.html?iref=rss-c142

Eco-Worthy’s 48V 100Ah server-rack lithium battery comes as a six-pack delivering about 30.72 kWh of usable energy in a standard 3U-compatible full-metal enclosure. Each battery pack includes a 100A battery management system, wireless connectivity and smart monitoring features for system diagnostics and safety, and multi-layer protection intended for off-grid, backup, or rack-mounted energy storage deployments.
4679.99
https://www.dealnews.com/Eco-Worthy-4

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Oukitel P2001 Plus is a high-capacity portable power station featuring a 2,048 Wh LiFePO4 battery and an integrated battery management system for safe, long-life performance. It provides 2,400 W continuous AC output with 4,800 W peak surge capability, supports up to 1,800 W AC input for fast recharging and accepts up to 500 W of solar input for off-grid charging. The unit is designed for heavy-duty backup power, camping, and emergency use where multiple AC and DC outputs are required.', price=569.0, url='https://www.dealnews.com/Oukitel-P2001-Plus-Portable-Power-Station-2048-Wh-2400-W-for-569-free-shipping/21803780.html?iref=rss-c142'), Deal(product_description='Eco-Worthy 48V 100Ah server-rack lithium battery kit consists of six 48V 100Ah modules packaged as a 30.72 kWh stack designed to fit standard 3U server cabinets. The pack includes smart monitoring and wireless connectivity, a durable full-metal enclosure, and a 100A battery managem

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [25]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
08:54:49 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
08:54:51 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
